## Getting back to the ranking of Features by Correlation

In [1]:
import pandas as pd 
import os 

# print(os.getcwd())
os.chdir("../data/")
# print(os.getcwd())

psalms = pd.read_csv("offical_psalms_data.csv")
psalms

/Users/caden/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/caden/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,Unnamed: 0,text_Psalter,psalm_num,verse,neg,neu,pos,compound,polarity,subjectivity,...,syllable_count,avg_syllables_per_word,polysyllable_count,num_sentences,avg_words_sent,kathisma_num,bertTopic,vader_category_Negative,vader_category_Neutral,vader_category_Positive
0,0,0,1,Blessed is the man Who walks not in the counse...,0.015,0.920,0.064,0.7783,0.000000,0.000000,...,163,1.225564,8,6,22.166667,1.0,0,0,0,1
1,2,0,2,Why do the nations rage And the people meditat...,0.120,0.787,0.093,-0.6486,-0.153333,0.400000,...,255,1.281407,12,12,16.583333,1.0,3,1,0,0
2,4,0,3,A psalm by David when he fled from the face of...,0.089,0.786,0.126,0.8020,0.048889,0.437778,...,183,1.220000,7,8,18.750000,1.0,1,0,0,1
3,6,0,4,For the End in psalms an ode by David You hear...,0.092,0.748,0.159,0.8919,0.178571,0.600000,...,189,1.145455,4,8,20.625000,1.0,1,0,0,1
4,8,0,5,For the End concerning the inheritance a psalm...,0.130,0.712,0.157,0.7774,0.075000,0.593750,...,309,1.245968,14,12,20.666667,1.0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296,591,1,146,Alleluia of Aggeus and Zacharias Praise the Lo...,0.095,0.765,0.141,0.8417,0.157407,0.481481,...,232,1.260870,10,11,16.727273,20.0,2,0,0,1
297,593,1,147,Alleluia of Aggeus and Zacharias Praise the Lo...,0.000,0.794,0.206,0.9808,-0.200000,0.650000,...,156,1.300000,4,9,13.333333,20.0,-1,0,0,1
298,595,1,148,Alleluia of Aggeus and Zacharias Praise the Lo...,0.018,0.749,0.233,0.9946,0.116667,0.316667,...,258,1.277228,10,14,14.428571,20.0,2,0,0,1
299,597,1,149,Let Israel be glad in Him who made him And let...,0.017,0.759,0.224,0.9870,0.265455,0.664848,...,177,1.282609,6,9,15.333333,20.0,2,0,0,1


In [2]:
# calculating correlations 
corr = psalms.corr(numeric_only=True)#['berTopic']

# pulling the column of correlations to the text - 'Bible' or 'Psalter'
correlations = corr['text_Psalter'].sort_values(key=abs, ascending=False)

correlations = pd.DataFrame(correlations)
correlations = correlations[2:]
# correlations

### Preparing the Data

In [3]:
# from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score


X = psalms[['neg', 'neu', 'pos', 'compound', 'polarity', 'subjectivity', 'emotional_score', 'fk_score',
       'gf_score', 'smog_score', 'word_count', 'avg_word_len', 'syllable_count', 'avg_syllables_per_word', 
       'polysyllable_count', 'num_sentences', 'avg_words_sent', 'kathisma_num', 'bertTopic',
       'vader_category_Negative', 'vader_category_Neutral', 'vader_category_Positive']]

y = psalms['text_Psalter']

#Splitting the dataset to train data and test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**---**
# Overall Goal of This Notebook
---
In the `modeling` notebook, the top 5 most correlated features proved to be the most successful for building a perceptron model. I am going to work at engineering the model to be the best that it can for the final work to the thesis and paper, as a whole.

## The best model from the `modeling` Notebook

In [4]:
def perceptron_trial(n=2):

    features = correlations[:n].index.tolist()
    X = psalms[features]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    perceptron = Perceptron(max_iter=100, eta0=0.1, random_state=42)
    perceptron.fit(X_train, y_train)

    y_pred = perceptron.predict(X_test)

    class_report = classification_report(
        y_test, y_pred, output_dict=True, zero_division=0
    )

    # Extract accuracy
    accuracy = class_report.pop("accuracy")

    # Convert remaining metrics into a DataFrame
    report_df = pd.DataFrame(class_report).T

    # Add accuracy as a column
    report_df["accuracy"] = accuracy
    report_df["features_used"] = str(features)
    report_df["n_features"] = n

    return report_df


result_df = perceptron_trial(n=5)
display(result_df)

,precision,recall,f1-score,support,accuracy,features_used,n_features
0,0.714286,0.322581,0.444444,31.0,0.590164,"['subjectivity', 'compound', 'neg', 'vader_cat...",5
1,0.553191,0.866667,0.675325,30.0,0.590164,"['subjectivity', 'compound', 'neg', 'vader_cat...",5
macro avg,0.633739,0.594624,0.559885,61.0,0.590164,"['subjectivity', 'compound', 'neg', 'vader_cat...",5
weighted avg,0.635059,0.590164,0.557992,61.0,0.590164,"['subjectivity', 'compound', 'neg', 'vader_cat...",5


The current **F1-Score** for the **macro-average** is a `0.559885`.

My goal is to get this metric as high as possdible, but hopefully above, or as close to, a `0.6`. There definetely is a difference between a Psalm from the Bible and the exact same Psalm from the Psalter, but they are not different enough for a model to acheive perfect accuaracy with the data that I have dervied in this research. 

The 5 features that will be focused on for this are the following: 

> - subjectivity 
> - compound
> - neg
> - vader_category_Negative
> - vader_category_Positive	

There are 5 different features to expereiment with and the first avenue that is going to be tried is turning them into a set of different combinatinos that try to get away from just a Perceptron on linear relationship.  

Adding every combination of 5 choose two to the 5 regular features. This is to see if there is an improved model. 

In [5]:
target_features = correlations[:5].index.tolist()
target_features

['subjectivity',
 'compound',
 'neg',
 'vader_category_Negative',
 'vader_category_Positive']

In [6]:
target_features_df = psalms[target_features]

In [7]:
def combo_columns(df, feat_1, feat_2):

    altered_feature = pd.DataFrame(index=df.index)

    col_name = f"{feat_1} & {feat_2}"

    altered_feature[col_name] = df[feat_1] * df[feat_2]

    new_features = df.copy()
    new_features[col_name] = altered_feature[col_name]

    return new_features, col_name

In [8]:
# testing 
# combo_columns(target_features_df, 'compound', 'neg')

(     subjectivity  compound    neg  vader_category_Negative  \
 0        0.000000    0.7783  0.015                        0   
 1        0.400000   -0.6486  0.120                        1   
 2        0.437778    0.8020  0.089                        0   
 3        0.600000    0.8919  0.092                        0   
 4        0.593750    0.7774  0.130                        0   
 ..            ...       ...    ...                      ...   
 296      0.481481    0.8417  0.095                        0   
 297      0.650000    0.9808  0.000                        0   
 298      0.316667    0.9946  0.018                        0   
 299      0.664848    0.9870  0.017                        0   
 300      0.650000    0.9930  0.000                        0   
 
      vader_category_Positive  compound & neg  
 0                          1        0.011674  
 1                          0       -0.077832  
 2                          1        0.071378  
 3                          1        0

---
#### Function to engineer new columns to input as "new_features" to a perceptron model. 
Things to add: 
- passing in the entrie df of features from the `combo_columns(feat_1, feat_2)`

In [54]:
def perceptron_trial(X, new_feature):
    
    # Separate target and predictors
    y = df['text_Psalter']
    X = df.drop(columns=['text_Psalter'])

    # Keep numeric predictors only
    X = X.select_dtypes(include='number')

    # Calculate number of features
    n = X.shape[1]

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    perceptron = Perceptron(max_iter=100, eta0=0.1, random_state=42)
    perceptron.fit(X_train, y_train)
    
    
    y_pred = perceptron.predict(X_test)

    class_report = classification_report(
        y_test, y_pred, output_dict=True, zero_division=0
    )

    # Extract accuracy
    accuracy = class_report.pop("accuracy")

    # Convert remaining metrics into a DataFrame
    report_df = pd.DataFrame(class_report).T

    # Add accuracy as a column
    report_df["accuracy"] = accuracy
    report_df["features_used"] = str(features)
    report_df["new_feature"] = new_feature
    report_df["n_features"] = n

    return report_df.reset_index().rename(columns={'index': 'classifier'})


result_df = perceptron_trial(target_features_df, 'test')
display(result_df)

NameError: name 'df' is not defined

With these updates lets integrate `combo_columns(feat_1, feat_2)` into a perceptron trial.

In [40]:
# max F1-score with  applying one new feature column per each perceptron trained
def find_best_f1(results_df):

    # Filter for macro average rows
    macro_results = results_df[
        results_df['classifier'] == 'macro avg'
    ]

    # Find index of highest F1-score
    best_idx = macro_results['f1-score'].idxmax()

    # Return entire row
    return results_df.loc[best_idx]


In [46]:
def new_trial(original_df, feature_1, feature_2, add_origin=True):

    new_features, new_col_name = combo_columns(
        original_df, feature_1, feature_2
    )
    
    # running a perceptron trial without the new column to include a a base line 
    if add_origin: 
        base = perceptron_trial(original_df, "baseline test")
        new_results = perceptron_trial(new_features, new_col_name)

        return pd.concat([base, new_results], axis=0, ignore_index=True)

    return perceptron_trial(new_features, new_col_name)
# Testing 
'''With the feature combination of  vader_category_Positive and coumpound as the 
   new engineered feature'''

results = new_trial(target_features_df, 'compound', 'neg', True) #boolean was used to debug find_best_f1()

# getting the best F1-Score 
    
find_best_f1(results)

classifier                                               macro avg
precision                                                 0.621528
recall                                                    0.594086
f1-score                                                  0.567376
support                                                       61.0
accuracy                                                  0.590164
features_used    ['subjectivity', 'compound', 'neg', 'vader_cat...
new_feature                                         compound & neg
n_features                                                       6
Name: 6, dtype: object

### Making Progress!

With only this trial of adding a the engineer feature of $ \textbf{compound * neg}$ we can already see an inrcrease of **F1-Score** for the **macro-average** to a `0.567376`. We can now see how adding every combination of the 5 targeted features affects the model. We can then see if adding 2 or more engineered feature columns contributes to the success of the `Perceptron Model`

In [55]:
import itertools

def feature_trial(eng_df, results_df):


    pairs = list(itertools.combinations(eng_df.columns, 2))

    for idx, pair in enumerate(pairs, 1):

        new_results = new_trial(eng_df, pair[0], pair[1])

        results_df = pd.concat(
            [results_df, new_results],
            axis=0,
            ignore_index=True
        )

    return results_df

In [56]:
results

,classifier,precision,recall,f1-score,support,accuracy,features_used,new_feature,n_features
0,0,0.714286,0.322581,0.444444,31.0,0.590164,"['subjectivity', 'compound', 'neg', 'vader_cat...",baseline test,5
1,1,0.553191,0.866667,0.675325,30.0,0.590164,"['subjectivity', 'compound', 'neg', 'vader_cat...",baseline test,5
2,macro avg,0.633739,0.594624,0.559885,61.0,0.590164,"['subjectivity', 'compound', 'neg', 'vader_cat...",baseline test,5
3,weighted avg,0.635059,0.590164,0.557992,61.0,0.590164,"['subjectivity', 'compound', 'neg', 'vader_cat...",baseline test,5
4,0,0.687500,0.354839,0.468085,31.0,0.590164,"['subjectivity', 'compound', 'neg', 'vader_cat...",compound & neg,6
5,1,0.555556,0.833333,0.666667,30.0,0.590164,"['subjectivity', 'compound', 'neg', 'vader_cat...",compound & neg,6
6,macro avg,0.621528,0.594086,0.567376,61.0,0.590164,"['subjectivity', 'compound', 'neg', 'vader_cat...",compound & neg,6
7,weighted avg,0.622609,0.590164,0.565748,61.0,0.590164,"['subjectivity', 'compound', 'neg', 'vader_cat...",compound & neg,6


In [57]:
full_results = pd.DataFrame() 

results_with_engineering = feature_trial(results, full_results)

TypeError: can't multiply sequence by non-int of type 'float'

In [44]:
find_best_f1(results_with_engineering)

classifier                                               macro avg
precision                                                 0.621528
recall                                                    0.594086
f1-score                                                  0.567376
support                                                       61.0
accuracy                                                  0.590164
features_used    ['subjectivity', 'compound', 'neg', 'vader_cat...
new_feature                                         compound & neg
n_features                                                       6
Name: 38, dtype: object

It looks like best comination of a pair of features to be combined and added as a feature is with features `subjectivity` and `compound`. these two colums, together, produced an F1-Score of **0.57377**. Lets conitue with trying to add another column with this new column: **subjectivity & compound**. 

In [14]:
new_df, new_col_name = combo_columns(target_features_df, 'subjectivity', 'compound')
# new_df
# new_df, new_col_name = new_df

In [15]:
full_results = pd.DataFrame() 

results_with_engineering = feature_trial(new_df, full_results)

The F1-score has decreased slightlty. Let see if we double the values of the `subjetivity & compound` column and see if thatdoes better that what is above. 

--- 
Reengieeing the perceptron function to accepted a new built set of features